# 04 Scenario Simulation & Strategic Impact

**Context:**
Notebook 03 statistically validated findings, proving that margin destruction is systemic and driven by specific structural failures (zombie SKUs, extreme profit concentration, chronic operational drag).

**Purpose:**
This notebook translates those validated audit findings into quantified financial projections under three operational stress scenarios. I will establish the current baseline, simulate the elimination of "Cut Candidates," stress-test the core profit drivers, and project the cost of compounded operational failure.

## 1. Environment Setup
Load the classified catalog data and the ground truth operational metrics.

In [1]:
import pandas as pd
import numpy as np

# Load data
classifications_path = '../data/processed/quadrant_classifications.csv'
ground_truth_path = '../data/processed/ground_truth_master.csv'

# Read CSVs
df_class = pd.read_csv(classifications_path)
df_gt = pd.read_csv(ground_truth_path)

# Merge the quadrant and risk score data into the ground truth dataframe
# This gives all revenue/profit data + the operational segments in one place
df = pd.merge(
    df_gt,
    df_class[['product_card_id', 'quadrant', 'composite_risk_score']],
    on='product_card_id',
    how='left'
)

print(f"Simulation dataset loaded: {df.shape[0]} products ready.")

Simulation dataset loaded: 118 products ready.


## 2. Baseline Establishment
Before simulating any operational changes, I must lock the current state as reference point. This baseline quantifies exactly what the business is currently generating, and critically, how much of that profit is reliant on the 8 core SKUs identified in Phase 3.

In [2]:
# Calculating Macro Baselines
baseline_revenue = df['total_revenue'].sum()
baseline_profit = df['total_profit'].sum()
baseline_margin = baseline_profit / baseline_revenue
total_skus = len(df)

# Isolate the Core 8 SKUs (Identified via upper IQR bound in Notebook 03)
core_8_df = df.sort_values(by='total_profit', ascending=False).head(8)
core_8_profit = core_8_df['total_profit'].sum()
core_8_revenue = core_8_df['total_revenue'].sum()
core_8_profit_pct = core_8_profit / baseline_profit

# Print Clean Summary Table
print("CURRENT STATE BASELINE")
print(f"Total Active SKUs:          {total_skus}")
print(f"Total Catalog Revenue:      ${baseline_revenue:,.2f}")
print(f"Total Catalog Profit:       ${baseline_profit:,.2f}")
print(f"Average Net Margin:         {baseline_margin:.2%}\n")
print("PROFIT CONCENTRATION")
print(f"Core 8 SKU Profit:          ${core_8_profit:,.2f}")
print(f"Core 8 SKU Revenue:         ${core_8_revenue:,.2f}")
print(f"Core Reliance Metric:       {core_8_profit_pct:.1%} of Total Profit")

CURRENT STATE BASELINE
Total Active SKUs:          118
Total Catalog Revenue:      $35,214,428.98
Total Catalog Profit:       $3,806,420.63
Average Net Margin:         10.81%

PROFIT CONCENTRATION
Core 8 SKU Profit:          $3,216,447.46
Core 8 SKU Revenue:         $29,834,820.73
Core Reliance Metric:       84.5% of Total Profit
